# Feature engineering — Silver → feature set (Gold)

Builds the model-ready feature table from Silver and saves it as
`wine_features.parquet`, keyed by `wine_id`. Downstream modelling
notebooks load this instead of re-deriving features.

**Scope (leakage-safe only).** This notebook holds deterministic
transforms that do *not* peek at the target or the train/test split:
ordinal encoding (a fixed label map), `log_retail`, and pass-through
numerics. Target encoding and standardisation are intentionally **not**
here — they must be fit on the training split inside the model notebook,
otherwise test information leaks into the features.

In [ ]:
import numpy as np
import pandas as pd
import itables
from itables import show
from sklearn.preprocessing import OrdinalEncoder

itables.options.columnDefs = [{"className": "dt-left", "targets": "_all"}]

SILVER_PATH   = r"..\..\.data\wine_reviews_silver.parquet"
FEATURES_PATH = r"..\..\.data\wine_features.parquet"

df = pd.read_parquet(SILVER_PATH)
print(f"Silver shape: {df.shape}")
assert df["wine_id"].is_unique, "wine_id must be unique"

## Categorical encoding (ordinal)

Leakage-free label map. `encoded_missing_value=-1` keeps NaNs as their
own code. Moved here from the old `04_baseline.ipynb`.

In [ ]:
cat_cols = ["country", "wine_type", "state", "appellation", "varietal_label", "company"]

enc = OrdinalEncoder(encoded_missing_value=-1)
df[[f"{c}_ord" for c in cat_cols]] = enc.fit_transform(df[cat_cols])

show(df[[c for col in cat_cols for c in (col, f"{col}_ord")]].value_counts().reset_index(), maxBytes="2MB")

## Engineered numerics

- `log_retail` — model on the log of the right-skewed price target.
- `age_at_review` — bottle age when reviewed: `date_of_review` year − `vintage`.
  Non-vintage (NV) wines have no `vintage` (NaN), so their age is left **null**
  rather than fabricated.

In [ ]:
df["log_retail"] = np.log(df["retail"])

# Bottle age at review. vintage is NaN for NV wines → age_at_review stays NaN.
review_year = pd.to_datetime(df["date_of_review"], errors="coerce").dt.year
df["age_at_review"] = review_year - df["vintage"]

print(f"age_at_review null: {df['age_at_review'].isna().sum():,} "
      f"(of which NV: {df['is_nv'].sum():,})")
df[["retail", "log_retail", "vintage", "age_at_review"]].describe().T

## Assemble & save feature set

`wine_id` + numerics + ordinal codes + target. One row per wine, joins
to any other feature table (keywords, embeddings) on `wine_id`.

In [ ]:
numeric_features = ["rating", "alcohol", "bottle_size", "vintage", "case_production", "age_at_review"]
ordinal_features = [f"{c}_ord" for c in cat_cols]
target_cols      = ["retail", "log_retail"]

feature_cols = ["wine_id"] + numeric_features + ordinal_features + target_cols
wine_features = df[feature_cols].copy()

wine_features.to_parquet(FEATURES_PATH, index=False)
print(f"Saved {wine_features.shape[0]:,} rows x {wine_features.shape[1]} cols -> {FEATURES_PATH}")
wine_features.head()